# mBERT and XLM-R: combined multilingual emotion run

Fine-tune mBERT on Google Colab with a GPU using the same seven emotion labels, Hindi-English plus English-French data, and exact 16,000/2,000/2,000 frozen split as the installed combined XLM-R model. This is supervised classifier fine-tuning, not language-model pretraining. Select **Runtime > Change runtime type > T4 GPU**, then run all cells and upload `mbert_study.zip` when prompted.

The dataset is synthetic and template-based. The held-out test split is intentionally the same one used by XLM-R, so this is a paired model comparison, not an independent evaluation. No independent human-labelled evaluation set is present in the project. Computed overlap counts report zero cross-split group, template-family and normalized-duplicate overlaps; this does not establish statistical independence. Inspect the bundled audit before interpreting results.


In [ ]:
from pathlib import Path
import hashlib, json, os, subprocess, sys, zipfile
from google.colab import files, drive
uploaded=files.upload()
zips=[n for n in uploaded if n.lower().endswith(".zip")]
assert len(zips)==1, "Upload exactly one generated training ZIP"
base=Path("/content").resolve()
with zipfile.ZipFile(zips[0]) as z:
    assert sum(i.file_size for i in z.infolist()) < 512*1024**2
    for i in z.infolist():
        assert "\\" not in i.filename and (base/i.filename).resolve().is_relative_to(base)
        assert (i.external_attr>>16)&0o170000 != 0o120000
    z.extractall(base)
PACKAGE=base/"hinglish_project_training"
m=json.loads((PACKAGE/"MANIFEST.json").read_text())
for name,digest in m["sha256"].items():
    p=(PACKAGE/name).resolve(); assert p.is_relative_to(PACKAGE)
    assert hashlib.sha256(p.read_bytes()).hexdigest()==digest, name
PROJECT=PACKAGE/"source"; os.chdir(PROJECT)
subprocess.run([sys.executable,"-m","pip","install","-e",".[model]"],check=True)
import torch
assert torch.cuda.is_available(), "Enable Runtime > Change runtime type > T4 GPU, then restart from this cell"
print("GPU:",torch.cuda.get_device_name(0))


In [ ]:
JOB = "mbert_combined"
EPOCHS = 3
assert JOB == "mbert_combined" and EPOCHS == 3
drive.mount("/content/drive")
STORAGE = Path("/content/drive/MyDrive/HinglishEmotionProject/MBERTStudy")
run = STORAGE / JOB
run.mkdir(parents=True, exist_ok=True)
DATA = PACKAGE / "data/all_splits.csv"
AUDIT = PACKAGE / "data/dataset_audit.json"
contract = json.loads((PACKAGE / "study/study_contract.json").read_text())
baseline = json.loads((PACKAGE / "study/xlmr_baseline_metrics.json").read_text())
baseline_metadata = json.loads((PACKAGE / "study/xlmr_baseline_metadata.json").read_text())
assert hashlib.sha256(json.dumps(baseline, sort_keys=True).encode()).hexdigest() == contract["baseline_metrics_sha256"]
assert hashlib.sha256(json.dumps(baseline_metadata, sort_keys=True).encode()).hexdigest() == contract["baseline_metadata_sha256"]
assert baseline_metadata["training_status"] == "complete"
assert baseline_metadata["task"] == "emotion" and baseline_metadata["labels"] == contract["labels"]
assert baseline_metadata["dataset_sha256"] == contract["dataset_sha256"]
assert baseline_metadata["split_sha256"] == contract["split_sha256"]


## Verify the baseline and freeze the split

The notebook checks the dataset hash, split hash, test identities, labels, split sizes and training settings against the installed XLM-R metadata before training. The current data must match exactly. It then saves the frozen split manifest to Drive, so a resumed run cannot silently use a different split.


In [ ]:
from training.data_workflow import prepare_data
def test_fingerprint(frame):
    rows = sorted((str(row.row_id), str(row.text), str(row.emotion), str(row.language_pair))
                  for row in frame.itertuples(index=False))
    return hashlib.sha256(json.dumps(rows, ensure_ascii=False, separators=(",", ":")).encode("utf-8")).hexdigest()
prepared = prepare_data(DATA, "emotion", "raw", 42, run / "frozen_split.csv", AUDIT,
                        "all", allow_synthetic=True)
assert prepared["dataset_sha256"] == contract["dataset_sha256"], "Dataset changed"
assert prepared["manifest_sha256"] == contract["split_sha256"], "Frozen split changed"
assert test_fingerprint(prepared["parts"][2]) == contract["test_sha256"], "Held-out test set changed"
sizes = dict(zip(("train", "validation", "test"), (len(part) for part in prepared["parts"])))
assert sizes == contract["split_sizes"] == {"train": 16000, "validation": 2000, "test": 2000}
print("Verified data and splits:", sizes)


## Train mBERT on the GPU

The run uses raw text, max length 128, batch size 2, gradient accumulation 8, learning rate 2e-5, seed 42, balanced class weights and fp16. It fine-tunes the full encoder and emotion head for up to three epochs; validation macro-F1 selects the checkpoint with patience 2. Drive stores the frozen split and epoch recovery state. If Colab disconnects, reopen this notebook, upload the same ZIP, keep the same job folder, and run all cells again to resume.

Model cards: [mBERT](https://huggingface.co/google-bert/bert-base-multilingual-cased), [XLM-R](https://huggingface.co/FacebookAI/xlm-roberta-base). CPU training is disabled.


In [ ]:
command = [sys.executable, "-u", "-m", "training.compare_encoders",
    "--task", "emotion", "--models", "mbert", "--data", str(DATA), "--language-pair", "all",
    "--output-root", str(run / "models"), "--split-manifest", str(run / "frozen_split.csv"),
    "--epochs", str(EPOCHS), "--batch-size", "2", "--gradient-accumulation", "8",
    "--max-length", "128", "--learning-rate", "2e-5", "--precision", "fp16",
    "--text-mode", "raw", "--seed", "42", "--split-seed", "42", "--resume",
    "--data-audit", str(AUDIT), "--allow-synthetic"]
subprocess.run(command, cwd=PROJECT, check=True)


## Compare and export

The comparison cell refuses metrics from another dataset, split, label set, model, or setting. Accuracy is the share of correct predictions; macro-F1 gives equal weight to each emotion. Both scores are from the shared synthetic test set and do not establish real-world performance. Test scores must not be used to select training settings.


In [ ]:
from training.cloud_bundle import export_checkpoint
from IPython.display import display
import pandas as pd
model_dir = run / "models/mbert_emotion"
metrics = json.loads((model_dir / "metrics.json").read_text())
assert metrics["base_model"] == contract["base_model"]
assert metrics["dataset_sha256"] == contract["dataset_sha256"]
assert metrics["split_sha256"] == contract["split_sha256"]
assert metrics["labels"] == contract["labels"]
assert metrics["split_sizes"] == contract["split_sizes"]
for key, value in contract["settings"].items():
    assert metrics[key] == value, f"Training setting changed: {key}"
assert test_fingerprint(prepared["parts"][2]) == contract["test_sha256"]
rows = []
for name, item in (("XLM-R baseline", baseline), ("mBERT", metrics)):
    rows.append({"model": name, "best_validation_macro_f1": item["best_validation_macro_f1"],
                 "test_accuracy": item["test"]["accuracy"], "test_macro_f1": item["test"]["macro_f1"],
                 "test_rows": item["split_sizes"]["test"]})
display(pd.DataFrame(rows))
print("Shared synthetic test set; not an independent human evaluation.")
export_path = STORAGE / "exports/mbert_combined.zip"
export_checkpoint(model_dir, export_path)
print("Download this inference checkpoint:", export_path)
files.download(str(export_path))
